# Flags: demo on the running example

Concept: `overview/Deriving Locations.md` (outside this repo), section "Flags (whole log)".
Code: `ocelescope_module_location_based_ocpm.derivation.flags`; this notebook only imports and calls its functions.
Further detail: the docstrings (e.g. `help(f3_mixed_pickup)`) and `docs/resources/location-derivation.md`, section 6.
The location derivation itself is demonstrated in `notebooks/location_derivation.ipynb`.

| flag | function | flags | idea |
|---|---|---|---|
| F1 partial movement | `f1_partial_movement` | events | a movement at which some handling unit was already at the target |
| F2 never located | `f2_never_located` | handling units | no event of the handling unit has a provided location |
| F3 mixed pickup | `f3_mixed_pickup` | events | the moved handling units of a movement came from different places |
| F4 location gap | `f4_location_gap`, `f4_location_gaps` | handling units | a provided pickup location differs from the last known location |
| F5 movement disagreement | `f5_movement_disagreement` | events | (a) a location change without movement, (b) a movement without location change |

Flags never block anything; they are always computed on the whole log (page filters do not apply) and at the selected level $k$: $loc = loc_k$, $ploc = ploc_k$, $before = before_{loc_k}$.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/flags.ipynb
```

In [ ]:
import json
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.config import DerivationConfig
from ocelescope_module_location_based_ocpm.derivation.flags import (
    at_target,
    f1_partial_movement,
    f2_never_located,
    f3_mixed_pickup,
    f4_location_gap,
    f4_location_gaps,
    f5_movement_disagreement,
)
from ocelescope_module_location_based_ocpm.derivation.pipeline import compute_flags, derive

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"

# Show whole frames (the running example has 28 relations).
pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250);

## 1. Input: the derived relations

The flags need, per hu-event relation, `is_movement`, `uloc`, `puloc`, $before_{loc_k}$ and $ploc_k$.
The saved level files (`data/running_example/derived/`) hold only `loc` and `ploc`, so the notebook derives the locations with `derive` (see `notebooks/location_derivation.ipynb`).
The running example's configuration selects Level 3, so all flags below use `before_3` and `ploc_3`.

In [ ]:
ocel = OCEL.read(EXAMPLE / "running_example.sqlite")
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
derivation = derive(ocel, config)
relations = derivation.relations
LEVEL = derivation.level
LEVEL

### Already at the target: `at_target(level)`

F1, F3 and F5 (b) ask whether a handling unit was already at the target of an event: $ploc_k((e, hu)) = uloc(e)$.
$\bot$ counts as a change, i.e. if $ploc_k$ or $uloc$ is $\bot$ the handling unit is *not* at the target.
`at_target(level)` is a polars expression (true/false) on the relations frame; below it is added as a column, for the movement events and the other events with a provided location.

In [ ]:
relations.filter(pl.col("is_movement") | pl.col("uloc").is_not_null()).select(
    "event", "hu", "activity", "is_movement", "uloc", "puloc", f"before_{LEVEL}", f"ploc_{LEVEL}",
    at_target(LEVEL).alias("at_target"),
).sort("event", "hu")

## 2. F1 partial movement: `f1_partial_movement(relations, level)`

A movement event $e \in M$ is flagged if at least one of its handling units was already at the target ($ploc_k = uloc(e)$).

| parameter | meaning |
|---|---|
| `relations` | relations frame with columns event, is_movement, uloc, ploc_k |
| `level` | selected level $k$ |

Returns the flagged event ids, sorted.

Example: at e11 (crane move to A2) both c1 and c2 are already at A2; at e12 (crane move to B1) c3 is already at B1 while c2 comes from A2.

In [ ]:
f1_partial_movement(relations, LEVEL)

## 3. F2 never located: `f2_never_located(relations)`

A handling unit is flagged if none of its events has a provided location ($uloc$).
It does not depend on the level (no `level` parameter): derived levels may still locate such a handling unit, e.g. Level 3 via transformation events.

Parameter: relations frame with columns hu, uloc. Returns the flagged handling units, sorted.

Example: c4 (events e15, e16) and c5 (e16, e17) never have a provided location; c4 is nevertheless located at Level 3 (B1, via the cut e15).

In [ ]:
f2_never_located(relations)

## 4. F3 mixed pickup: `f3_mixed_pickup(relations, level)`

A movement event without a provided pickup location is flagged if the handling units it actually moves had different previous locations ($ploc_k$).
Handling units already at the target are left out (they were not moved), and $\bot$ counts as one previous location of its own.
Events with a provided pickup location are never flagged (their $ploc_k$ is the pickup location for all handling units).

| parameter | meaning |
|---|---|
| `relations` | relations frame with columns event, is_movement, uloc, puloc, ploc_k |
| `level` | selected level $k$ |

Returns the flagged event ids, sorted.

Example: e8 (load truck) picks up c1 from B1, c2 from A2 and c3 from $\bot$: three different previous locations.

In [ ]:
f3_mixed_pickup(relations, LEVEL)

## 5. F4 location gap: `f4_location_gaps(relations, level)` and `f4_location_gap(relations, level)`

A gap is a hu-event relation where the provided pickup location $puloc(e)$ differs from the handling unit's last known location $before_{loc_k}$.
A first known location ($before_{loc_k} = \bot$) is not a gap.
The handling unit must have got from its last known location to the pickup location without any recorded movement.

- `f4_location_gaps` lists the gaps (columns hu, event, last_known, pickup), for exploring them;
- `f4_location_gap` returns the flagged handling units (those with at least one gap), sorted.

Parameters of both: relations frame with columns hu, event, puloc, before_k; selected level $k$.

Example: c1 was last at A2 (e11) but is picked up at A1 by e13.

In [ ]:
f4_location_gaps(relations, LEVEL)

In [ ]:
f4_location_gap(relations, LEVEL)

## 6. F5 movement disagreement: `f5_movement_disagreement(relations, level)`

Events whose classification as movement or non-movement disagrees with the locations:
- (a) *location change without movement*: an event with a provided location that is not a movement ($e \in dom(uloc) \setminus M$), where some handling unit's $before_{loc_k}$ is neither $\bot$ nor $uloc(e)$;
- (b) *movement without location change*: a movement event $e \in M$ where every handling unit was already at the target ($ploc_k = uloc(e)$).

| parameter | meaning |
|---|---|
| `relations` | relations frame with columns event, is_movement, uloc, before_k, ploc_k |
| `level` | selected level $k$ |

Returns columns event, kind (`"a"` or `"b"`), sorted by event.

Example: (a) e7 (quality check at B1) is not a movement, but c1 was last at A2; (b) at e11 (crane move to A2) both handling units were already at A2.
Note that e11 is also F1: (b) is the case of F1 where *all* handling units were at the target.

In [ ]:
f5_movement_disagreement(relations, LEVEL)

## 7. All flags in one call: `compute_flags(relations, level)` and `derive`

`compute_flags` (in `pipeline.py`) runs all five and returns a `Flags` object with attributes `f1` to `f5` (`f5` is the frame with the kinds).
`derive` calls it at the configured level and stores the result in `Derivation.flags` (None if there are blocking errors).
The result matches the concept's flags table (`data/running_example/expected/flags.csv`).

In [ ]:
flags = derivation.flags
flags

In [ ]:
expected = pl.read_csv(EXAMPLE / "expected" / "flags.csv")
actual = pl.concat(
    [
        pl.DataFrame({"flag": "F1", "unit": "event", "id": flags.f1}),
        pl.DataFrame({"flag": "F2", "unit": "handling unit", "id": flags.f2}),
        pl.DataFrame({"flag": "F3", "unit": "event", "id": flags.f3}),
        pl.DataFrame({"flag": "F4", "unit": "handling unit", "id": flags.f4}),
        pl.DataFrame({"flag": "F5", "unit": "event", "id": flags.f5["event"]}),
    ]
)
actual.sort(pl.all()).equals(expected.sort(pl.all()))